<a href="https://colab.research.google.com/github/marachiramya159-19/-Implementation-of-random-forest/blob/main/Transfer_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
import os

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!ls /content/drive/MyDrive/YK/AI/CVE/

 Architecture_CNN.ipynb  'Image Classification'  'Transfer learning'
 ASSIGNMENT		 'Image Segmentation'	  tree-clipart.png
 CNN_Operations.ipynb	  Keras.ipynb
 CNN_RGB.ipynb		 'Object detection'


In [ ]:
!ls /content/drive/MyDrive/YK/AI/CVE/Transfer\ learning/TRANSFER\ LEARNING/mammals


In [ ]:
data = "/content/drive/MyDrive/YK/AI/CVE/Transfer learning/TRANSFER LEARNING/mammals"
print(data)

/content/drive/MyDrive/YK/AI/CVE/Transfer learning/TRANSFER LEARNING/mammals


In [ ]:
img_size = (224, 224)  # Standard size for most pretrained CNNs
batch_size = 32

In [ ]:
datagen = ImageDataGenerator(rescale=1./255, validation_split=0.2)

train_data = datagen.flow_from_directory(
    data,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='training'
)

Found 11033 images belonging to 45 classes.


In [ ]:
val_data = datagen.flow_from_directory(
    data,
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation'
)

Found 2734 images belonging to 45 classes.


In [ ]:
base = MobileNetV2(include_top=False, weights='imagenet', input_shape=(224,224,3)) #include_top = False - removing th original layers
base.trainable = False  # freeze base layers

x = GlobalAveragePooling2D()(base.output)
output = Dense(train_data.num_classes, activation='softmax')(x)

model = Model(inputs=base.input, outputs=output)
model.compile(optimizer=Adam(0.0001), loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,315,629 (8.83 MB)

 Trainable params: 57,645 (225.18 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [ ]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=8
)

Epoch 1/8
345/345 ━━━━━━━━━━━━━━━━━━━━ 5296s 15s/step - accuracy: 0.3920 - loss: 2.7025 - val_accuracy: 0.6942 - val_loss: 1.6868
Epoch 2/8
345/345 ━━━━━━━━━━━━━━━━━━━━ 618s 2s/step - accuracy: 0.7831 - loss: 1.2134 - val_accuracy: 0.8168 - val_loss: 0.9488
Epoch 3/8
345/345 ━━━━━━━━━━━━━━━━━━━━ 583s 2s/step - accuracy: 0.8563 - loss: 0.7596 - val_accuracy: 0.8508 - val_loss: 0.6968
Epoch 4/8
345/345 ━━━━━━━━━━━━━━━━━━━━ 623s 2s/step - accuracy: 0.8839 - loss: 0.5740 - val_accuracy: 0.8691 - val_loss: 0.5764
Epoch 5/8
345/345 ━━━━━━━━━━━━━━━━━━━━ 587s 2s/step - accuracy: 0.8978 - loss: 0.4741 - val_accuracy: 0.8775 - val_loss: 0.5083
Epoch 6/8
345/345 ━━━━━━━━━━━━━━━━━━━━ 588s 2s/step - accuracy: 0.9087 - loss: 0.4091 - val_accuracy: 0.8873 - val_loss: 0.4608
Epoch 7/8
345/345 ━━━━━━━━━━━━━━━━━━━━ 591s 2s/step - accuracy: 0.9163 - loss: 0.3620 - val_accuracy: 0.8895 - val_loss: 0.4297
Epoch 8/8
345/345 ━━━━━━━━━━━━━━━━━━━━ 631s 2s/step - accuracy: 0.9250 - loss: 0.3262 - val_accuracy: 

In [ ]:
model.save("/content/drive/MyDrive/mammals_model.h5")

In [ ]:
print(train_data.class_indices)